# Python Finance Paper Replication: Empirical Asset Pricing via ML, GKX

# Step 1: Let's build the 94 characteristics

## 1) Compustat/CRSP variables

According to GHZ SAS code and their literature.

### a) Annual Compustat Data

Get: sale, revt, cogs, xsga, dp, xrd, xad, ib, ebitda, ebit, nopi, spi, pi, txp, ni, txfed, txfo, txt, xint, capx, oancf, dvt, ob, gdwlia, gdwlip, gwo, rect, act, che, ppegt, invt, at, aco, intan, ao, ppent, gdwl, fatb, fatl, lct, dlc, dltt, lt, dm, dcvt, cshrc, dcpstk, pstk, ap, lco, lo, drc, drlt, txdi, ceq, scstkc, emp, csho, sic, naics, prcc_f, fyear, datadate, gvkey

### b) Quaterly Compustat Data

Get = ibq, saleq, txtq, revtq, cogsq, xsgaq, atq, actq, cheq, lctq, dlcq, ppentq, prccq, cshoq, seqq, pstkq, atq, ltq, pstkrq, fyearq, fqtr, rdq, sic, ceqq, gvkey, datadate

### c) Monthly CRSP Stock File

Get = permno, mthcaldt, mthret, mthprc, shrout, mthvol

### d) CRSP Delisting Information

Get = permno, delistingdt, delret, delretmisstype, delactiontype, delstatustype, delreasontype

### e) CRSP/Compustat Merged (CCM) link table

Get = gvkey, lpermno, lpermco, linktype, linkprim, linkdt, linkenddt

## 2) Use SQL to get the data

Will use SQL to pull data. Could use the Tidy Finance library (https://www.tidy-finance.org/chapters/wrds-crsp-and-compustat.html) which uses SQL too.

In [1]:
# Connect to WRDS

import pandas as pd
import wrds

db = wrds.Connection(wrds_username='hcdb') 

Loading library list...
Done


In [ ]:
# Get the Fundamentals Annual data using SQL

funda = db.raw_sql("""
    SELECT c.gvkey, c.cik, c.sic, c.naics,
           f.datadate, f.fyear, f.sale, f.revt, f.cogs, f.xsga, f.dp, f.xrd, f.xad, f.ib, f.ebitda,
           f.ebit, f.nopi, f.spi, f.pi, f.txp, f.ni, f.txfed, f.txfo, f.txt, f.xint,
           f.capx, f.oancf, f.dvt, f.ob, f.gdwlia, f.gdwlip, f.gwo,
           f.rect, f.act, f.che, f.ppegt, f.invt, f.at, f.aco, f.intan, f.ao,
           f.ppent, f.gdwl, f.fatb, f.fatl,
           f.lct, f.dlc, f.dltt, f.lt, f.dm, f.dcvt, f.cshrc, f.dcpstk, f.pstk,
           f.ap, f.lco, f.lo, f.drc, f.drlt, f.txdi,
           f.ceq, f.scstkc, f.emp, f.csho,
           f.prcc_f
    FROM comp.funda AS f
    JOIN comp.company AS c
      ON f.gvkey = c.gvkey
    WHERE f.datadate >= '1975-01-01'
      AND f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.popsrc = 'D'
      AND f.consol = 'C'
""")

# select compustat variable from the fundamentals data (f) and the company reference (c)
# join them on gvkey
# use the correct data and the correct settings

In [9]:
display(funda.head(10))
display(list(funda.columns))
display(funda.shape)

,gvkey,datadate,fyear,cik,sic,naics,sale,revt,cogs,xsga,...,lco,lo,drc,drlt,txdi,ceq,scstkc,emp,csho,prcc_f
0,001000,1975-12-31,1975,<NA>,3089,<NA>,51.192,51.192,35.237,9.591,...,2.125,0.0,<NA>,<NA>,0.689,10.24,<NA>,1.054,2.098,4.375
1,001000,1976-12-31,1976,<NA>,3089,<NA>,66.414,66.414,45.277,12.396,...,2.485,0.0,<NA>,<NA>,0.237,14.106,<NA>,1.46,2.207,5.75
2,001000,1977-12-31,1977,<NA>,3089,<NA>,77.946,77.946,56.959,14.977,...,3.691,0.0,<NA>,<NA>,0.17,15.876,<NA>,1.741,2.226,9.25
3,001001,1978-12-31,1978,0000723576,5812,722,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,001001,1979-12-31,1979,0000723576,5812,722,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,001001,1980-12-31,1980,0000723576,5812,722,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,001001,1981-12-31,1981,0000723576,5812,722,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,001001,1982-12-31,1982,0000723576,5812,722,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,001001,1983-12-31,1983,0000723576,5812,722,25.395,25.395,6.31,16.435,...,0.706,0.0,<NA>,<NA>,0.144,7.823,<NA>,1.386,3.568,7.25
9,001001,1984-12-31,1984,0000723576,5812,722,32.007,32.007,8.171,20.628,...,0.788,0.0,<NA>,<NA>,0.275,8.962,<NA>,1.502,3.568,3.75


['gvkey',
 'datadate',
 'fyear',
 'cik',
 'sic',
 'naics',
 'sale',
 'revt',
 'cogs',
 'xsga',
 'dp',
 'xrd',
 'xad',
 'ib',
 'ebitda',
 'ebit',
 'nopi',
 'spi',
 'pi',
 'txp',
 'ni',
 'txfed',
 'txfo',
 'txt',
 'xint',
 'capx',
 'oancf',
 'dvt',
 'ob',
 'gdwlia',
 'gdwlip',
 'gwo',
 'rect',
 'act',
 'che',
 'ppegt',
 'invt',
 'at',
 'aco',
 'intan',
 'ao',
 'ppent',
 'gdwl',
 'fatb',
 'fatl',
 'lct',
 'dlc',
 'dltt',
 'lt',
 'dm',
 'dcvt',
 'cshrc',
 'dcpstk',
 'pstk',
 'ap',
 'lco',
 'lo',
 'drc',
 'drlt',
 'txdi',
 'ceq',
 'scstkc',
 'emp',
 'csho',
 'prcc_f']

(536477, 65)

In [11]:
# Get the Fundamental Quarterly data using SQL

fundq = db.raw_sql("""
    SELECT c.gvkey, c.sic,
           f.datadate, f.fyearq, f.fqtr,
           f.ibq, f.saleq, f.txtq, f.revtq, f.cogsq, f.xsgaq,
           f.atq, f.actq, f.cheq, f.lctq, f.dlcq, f.ppentq, f.prccq, f.cshoq,
           f.seqq, f.pstkq, f.ltq, f.pstkrq, f.rdq, f.ceqq
    FROM comp.fundq AS f
    JOIN comp.company AS c
      ON f.gvkey = c.gvkey
    WHERE f.datadate >= '1975-01-01'
      AND f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.popsrc = 'D'
      AND f.consol = 'C'
""")

In [12]:
display(fundq.head(10))
display(list(fundq.columns))
display(fundq.shape)

,gvkey,sic,datadate,fyearq,fqtr,ibq,saleq,txtq,revtq,cogsq,...,dlcq,ppentq,prccq,cshoq,seqq,pstkq,ltq,pstkrq,rdq,ceqq
0,001000,3089,1975-03-31,1975,1,0.19,11.295,0.168,11.295,<NA>,...,0.8,<NA>,3.0,2.092,10.307,0.414,<NA>,<NA>,1975-05-02,9.893
1,001000,3089,1975-06-30,1975,2,0.875,14.441,0.777,14.441,<NA>,...,0.0,<NA>,2.875,2.092,11.182,0.414,<NA>,<NA>,1975-07-16,10.768
2,001000,3089,1975-09-30,1975,3,0.613,13.122,0.543,13.122,<NA>,...,0.0,<NA>,4.0,2.094,11.8,0.414,<NA>,<NA>,1975-10-09,11.386
3,001000,3089,1975-12-31,1975,4,0.606,12.334,0.452,12.334,35.237,...,1.251,7.263,4.375,2.098,12.309,2.069,11.596,<NA>,1976-03-04,10.24
4,001000,3089,1976-03-31,1976,1,0.726,16.226,0.569,16.226,11.138,...,1.674,12.059,6.75,2.096,13.062,2.069,19.469,<NA>,1976-04-20,10.993
5,001000,3089,1976-06-30,1976,2,1.603,19.637,1.26,19.637,12.981,...,1.257,12.693,7.25,2.163,14.87,2.069,20.287,<NA>,1976-07-14,12.801
6,001000,3089,1976-09-30,1976,3,0.561,14.733,0.441,14.733,10.722,...,1.507,12.93,7.125,2.172,15.501,2.069,19.322,<NA>,1976-10-18,13.432
7,001000,3089,1976-12-31,1976,4,0.544,15.818,0.47,15.818,12.034,...,0.35,16.44,5.75,2.207,15.675,1.569,22.911,<NA>,1977-03-01,14.106
8,001000,3089,1977-03-31,1977,1,0.563,18.014,0.48,18.014,13.206,...,0.35,16.665,6.25,2.209,16.248,1.569,25.861,<NA>,1977-05-03,14.679
9,001000,3089,1977-06-30,1977,2,0.975,21.203,<NA>,21.203,15.286,...,0.35,16.438,6.5,2.223,17.271,1.569,24.822,<NA>,1977-07-21,15.702


['gvkey',
 'sic',
 'datadate',
 'fyearq',
 'fqtr',
 'ibq',
 'saleq',
 'txtq',
 'revtq',
 'cogsq',
 'xsgaq',
 'atq',
 'actq',
 'cheq',
 'lctq',
 'dlcq',
 'ppentq',
 'prccq',
 'cshoq',
 'seqq',
 'pstkq',
 'ltq',
 'pstkrq',
 'rdq',
 'ceqq']

(2034725, 25)

In [25]:
# Get the Monthly CRSP Stock File using SQL

msf = db.raw_sql("""
                 SELECT permno, mthcaldt, mthret, mthprc, shrout, mthvol,  primaryexch, sharetype
                 FROM crsp.msf_v2
                 WHERE mthcaldt >= '1975-01-01'
                 """)

In [26]:
display(msf.head(10))
display(list(msf.columns))
display(msf.shape)

,permno,mthcaldt,mthret,mthprc,shrout,mthvol,primaryexch,sharetype
0,10000,1986-01-31,0.707317,4.375,3680,177082.0,Q,NS
1,10000,1986-02-28,-0.257143,3.25,3680,82800.0,Q,NS
2,10000,1986-03-31,0.365385,4.4375,3680,107801.0,Q,NS
3,10000,1986-04-30,-0.098592,4.0,3793,95700.0,Q,NS
4,10000,1986-05-30,-0.222656,3.109375,3793,107362.0,Q,NS
5,10000,1986-06-30,-0.005025,3.09375,3793,106862.0,Q,NS
6,10000,1986-07-31,-0.080808,2.84375,3793,116305.0,Q,NS
7,10000,1986-08-29,-0.615385,1.09375,3793,304906.0,Q,NS
8,10000,1986-09-30,-0.057143,1.03125,3793,355100.0,Q,NS
9,10000,1986-10-31,-0.242424,0.78125,3843,190321.0,Q,NS


['permno',
 'mthcaldt',
 'mthret',
 'mthprc',
 'shrout',
 'mthvol',
 'primaryexch',
 'sharetype']

(4433007, 8)

In [18]:
# Get the CRSP Delisting Information using SQL ()

# Delisting returns capture the true gain/loss when a stock stops trading (e.g. bankruptcy), 
# so merging them in avoids survivorship bias that would otherwise make returns look artificially better.

dlret = db.raw_sql("""
                   SELECT permno, delistingdt, delret, delretmisstype, delactiontype, delstatustype, delreasontype
                   FROM crsp.stkdelists
                   """)

In [20]:
display(dlret.head(10))
display(list(dlret.columns))
display(dlret.shape)

,permno,delistingdt,delret,delretmisstype,delactiontype,delstatustype,delreasontype
0,10000,1987-06-11,0.0,NA,GDR,VCL,INSC
1,10001,2017-08-03,0.0,NA,MER,FPAY,UNAV
2,10002,2013-02-15,0.010906,NA,MER,FPAY,UNAV
3,10003,1995-12-15,-0.003648,NA,MER,FPAY,UNAV
4,10004,1986-01-17,<NA>,DM,GDR,VCL,OFFRE
5,10005,1991-07-11,0.0,NA,GDR,VCL,INSC
6,10006,1984-06-28,0.011601,NA,MER,FPAY,UNAV
7,10007,1990-10-16,0.0,NA,GDR,VCL,INSC
8,10008,1988-11-21,0.0,NA,GDR,VCL,SHLD
9,10009,2000-11-03,0.003774,NA,MER,FPAY,UNAV


['permno',
 'delistingdt',
 'delret',
 'delretmisstype',
 'delactiontype',
 'delstatustype',
 'delreasontype']

(29833, 7)

In [22]:
# Get the CRSP/Compustat Merged link table using SQL

ccmlink = db.raw_sql("""
                     SELECT gvkey, lpermno, lpermco, linktype, linkprim, linkdt, linkenddt
                     FROM crsp.ccmxpf_lnkhist
                     WHERE linktype IN ('LC','LU')
                     AND linkprim IN ('P','C')
                     """)

In [27]:
display(ccmlink.head(10))
display(list(ccmlink.columns))
display(ccmlink.shape)

,gvkey,lpermno,lpermco,linktype,linkprim,linkdt,linkenddt
0,001000,25881.0,23369.0,LU,P,1970-11-13,1978-06-30
1,001001,10015.0,6398.0,LU,P,1983-09-20,1986-07-31
2,001002,10023.0,22159.0,LC,C,1972-12-14,1973-06-05
3,001003,10031.0,6672.0,LU,C,1983-12-07,1989-08-16
4,001004,54594.0,20000.0,LU,P,1972-04-24,<NA>
5,001005,61903.0,11.0,LU,C,1973-01-31,1983-01-31
6,001007,10058.0,20.0,LU,C,1973-10-01,1979-01-30
7,001007,10058.0,20.0,LU,P,1979-01-31,1984-09-28
8,001008,10066.0,6331.0,LC,P,1983-08-25,1987-02-26
9,001009,10074.0,5271.0,LC,C,1982-01-18,1996-03-13


['gvkey', 'lpermno', 'lpermco', 'linktype', 'linkprim', 'linkdt', 'linkenddt']

(33324, 7)

In [30]:
# check if CRPS or Compustat tables names changed 

db.list_tables(library='crsp')
db.list_tables(library='comp')

['aco_amda',
 'aco_imda',
 'aco_indfnta',
 'aco_indfntq',
 'aco_indfntytd',
 'aco_indsta',
 'aco_indstq',
 'aco_indstytd',
 'aco_notesa',
 'aco_notesq',
 'aco_notessa',
 'aco_notesytd',
 'aco_pnfnda',
 'aco_pnfndq',
 'aco_pnfndytd',
 'aco_pnfnta',
 'aco_pnfntq',
 'aco_pnfntytd',
 'aco_transa',
 'aco_transq',
 'aco_transsa',
 'aco_transytd',
 'adsprate',
 'asec_amda',
 'asec_imda',
 'asec_notesa',
 'asec_notesq',
 'asec_transa',
 'asec_transq',
 'bank_aacctchg',
 'bank_adesind',
 'bank_afnd1',
 'bank_afnd2',
 'bank_afnddc1',
 'bank_afnddc2',
 'bank_afntind',
 'bank_funda',
 'bank_funda_fncd',
 'bank_fundq',
 'bank_fundq_fncd',
 'bank_iacctchg',
 'bank_idesind',
 'bank_ifndq',
 'bank_ifndytd',
 'bank_ifntq',
 'bank_ifntytd',
 'bank_names',
 'bank_namesq',
 'chars',
 'co_aacctchg',
 'co_aaudit',
 'co_acthist',
 'co_adesind',
 'co_adjfact',
 'co_afnd1',
 'co_afnd2',
 'co_afnddc1',
 'co_afnddc2',
 'co_afntind1',
 'co_afntind2',
 'co_ainvval',
 'co_amkt',
 'co_busdescl',
 'co_cotype',
 'co_f

## 3) Merging and cleaning dataset